# 09 — Publication tables, confidence intervals, and figures

This notebook performs **no training and no parameter selection**. It only aggregates frozen CSV/JSON artifacts. All primary quality statistics use the 2,000-image test split. Bootstrap intervals are per-image; predictability/detectability correlation uses image-cluster bootstrap.

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np, yaml
from rdhlab.publication import aggregate_test_results
from rdhlab.statistics import predictability_detectability_stats, cluster_bootstrap

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
out=Path('/workspace/results/publication'); out.mkdir(parents=True,exist_ok=True)
summary,feasibility=aggregate_test_results('/workspace/results/frozen_test/per_image.csv',out)
display(summary); display(feasibility)

In [ ]:
# Detectability table if notebook 07 has been completed.
det=Path('/workspace/results/detectors/test_detectability.csv')
if det.exists():
    ddf=pd.read_csv(det); ddf.to_csv(out/'table_detectability.csv',index=False); display(ddf)
else:
    print('Detector results not found yet; run notebook 07.')

In [ ]:
# Final test-set P-vs-D relationship using the frozen auxiliary risk model.
import joblib
from rdhlab.io import read_gray
risk=joblib.load('/workspace/results/models/block_risk.joblib')
manifest=pd.read_csv(config['dataset']['prepared_manifest']); test=manifest[manifest.split=='test'].reset_index(drop=True)
rows=[]
for i,row in test.iterrows():
    x=read_gray(row.path)
    for b in risk.score_image_blocks(x,str(row.source_id)):
        b['source_id']=str(row.source_id); rows.append(b)
    if (i+1)%250==0: print(i+1,'/2000')
bdf=pd.DataFrame(rows); stats=predictability_detectability_stats(bdf)
ci=cluster_bootstrap(bdf,'source_id',lambda z: predictability_detectability_stats(z)['spearman_rho'],n_resamples=int(config['statistics']['cluster_bootstrap_resamples']),seed=int(config['project']['seed']))
(Path(out/'predictability_detectability_test.json')).write_text(json.dumps({'statistics':stats,'spearman_cluster_bootstrap':ci},indent=2))
print(stats); print(ci)

In [ ]:
# Key ablation table: raster vs P-only vs D-only vs P+D at equal net payload.
per=pd.read_csv('/workspace/results/frozen_test/per_image.csv'); ok=per[per.feasible==True]
ablation=ok.groupby(['strategy','target_net_bpp']).agg(
    n=('source_id','count'), psnr_median=('psnr','median'), ssim_median=('ssim','median'),
    sideinfo_median=('sideinfo_bits','median'), used_blocks_median=('used_blocks','median'),
    selected_P=('selected_predictability_mean','mean'), selected_D=('selected_detectability_risk_mean','mean'),
    exact_recovery=('exact_image','mean')).reset_index()
ablation.to_csv(out/'table_ablation.csv',index=False); display(ablation)

In [ ]:
# Freeze audit for manuscript/reproducibility supplement.
audit={
 'experiment_version':config['project']['version'],
 'baseline_version':config['project']['baseline_version'],
 'payload_freeze':json.loads(Path('/workspace/config/frozen_payloads.json').read_text()),
 'allocator_freeze':json.loads(Path('/workspace/config/frozen_allocator.json').read_text()),
 'test_rows':int(len(per)), 'feasible_rows':int(ok.shape[0]),
 'all_feasible_exact_image':bool(ok.exact_image.all()), 'all_feasible_exact_message':bool(ok.exact_message.all()),
}
(out/'experiment_audit.json').write_text(json.dumps(audit,indent=2))
print(json.dumps(audit,indent=2))

Primary scientific claims should be drawn from these frozen outputs: exact reversibility, accounted net payload, P–D rank disagreement, equal-net-payload ablation, independent detector AUC/TPR@5%FPR, and resource cost. Cross-preprocessing results remain secondary robustness evidence.